#### <span style="color: deepskyblue;">Analiza dotycząca oferowanych powierzchni nieruchomości w ujęciu geograficznym   </span>


In [14]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as ss
import seaborn as sns
import missingno as msno
import plotly.express as px

%matplotlib inline 
%config InlineBackend.figure_format = 'retina'

df = pd.read_csv("../data/apartments_clean.csv", sep=';', encoding="cp1250")
df.head(1)

C:\Users\Grażyna\AppData\Local\Temp\ipykernel_4896\48487720.py:12: DtypeWarning: Columns (0: address) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/apartments_clean.csv", sep=';', encoding="cp1250")


,id,category,title,bathrooms,bedrooms,currency,fee,has_photo,pets_allowed,price,...,Playground,Pool,Refrigerator,Storage,TV,Tennis,View,Washer Dryer,Wood Floors,price_per_sqft
0,5668640009,housing/rent/apartment,One BR 507 & 509 Esplanade,1.0,1.0,USD,No,Thumbnail,Cats,2195.0,...,0,0,0,0,0,0,0,0,0,4.049815


Skróty stref czasowych użyte w tabeli df_state:
- EST (Wschodnia), 
- CST (Centralna), 
- MST (Górska), 
- PST (Pacyficzna), 
- AKST (Alaska), 
- HST (Hawaje). 

Plik grupujący stany USA wg różnych kategorii pozyskany jest z pomocą AI ze strony U.S. Census Bureau (Biuro Spisu Ludności USA).</BR> 
Oznaczenie „Tak” w kolumnach <span style="color: deepskyblue;">Stan rolniczy</span>, <span style="color: deepskyblue;">Stan przemysłowy</span>, <span style="color: deepskyblue;">Ważny turystycznie</span> wskazuje,</BR> że dany sektor stanowi kluczowy, wyróżniający się filar gospodarki lub tożsamości stanu na tle reszty kraju 

In [15]:
df_state = pd.read_csv("../data/przypisanie_stanowUSA.csv", sep=';', encoding="cp1250")
df_state.head(1)

,Stan,Skrot,Region,Strefa czasowa,Stan rolniczy,Stan przemyslowy,Wazny turystycznie
0,Alabama,AL,South,CST / EST,Tak,Tak,Nie


In [16]:

# dołącz nazwę stanu
df = df.merge(
    df_state[['Skrot', 'Stan']],
    left_on='state',
    right_on='Skrot',
    how='left'
)
# usuń pomocniczą kolumnę
df = df.drop(columns='Skrot')

# dołącz region
df = df.merge(
    df_state[['Skrot', 'Region']],
    left_on='state',
    right_on='Skrot',
    how='left'
)
# usuń pomocniczą kolumnę
df = df.drop(columns='Skrot')

# dołącz Strefa czasowa
df = df.merge(
    df_state[['Skrot', 'Strefa czasowa']],
    left_on='state',
    right_on='Skrot',
    how='left'
)
# usuń pomocniczą kolumnę
df = df.drop(columns='Skrot')

# dołącz Stan rolniczy
df = df.merge(
    df_state[['Skrot', 'Stan rolniczy']],
    left_on='state',
    right_on='Skrot',
    how='left'
)
# usuń pomocniczą kolumnę
df = df.drop(columns='Skrot')

# dołącz Stan przemyslowy
df = df.merge(
    df_state[['Skrot', 'Stan przemyslowy']],
    left_on='state',
    right_on='Skrot',
    how='left'
)
# usuń pomocniczą kolumnę
df = df.drop(columns='Skrot')

# dołącz Wazny turystycznie
df = df.merge(
    df_state[['Skrot', 'Wazny turystycznie']],
    left_on='state',
    right_on='Skrot',
    how='left'
)
# usuń pomocniczą kolumnę
df = df.drop(columns='Skrot')

df.head(1)

,id,category,title,bathrooms,bedrooms,currency,fee,has_photo,pets_allowed,price,...,View,Washer Dryer,Wood Floors,price_per_sqft,Stan,Region,Strefa czasowa,Stan rolniczy,Stan przemyslowy,Wazny turystycznie
0,5668640009,housing/rent/apartment,One BR 507 & 509 Esplanade,1.0,1.0,USD,No,Thumbnail,Cats,2195.0,...,0,0,0,4.049815,California,West,PST,Tak,Tak,Tak


In [ ]:
df['charakter'] = (
    np.where(df['Stan rolniczy']== "Tak" , 'Roln ', '') +
    np.where(df['Stan przemyslowy']== "Tak", 'Przem ', '') +
    np.where(df['Wazny turystycznie']== "Tak", 'Turyst', '')
)

df['charakter'] = df['charakter'].str.strip()
df.head(10)

In [13]:
#df["bin_square_feet"] = pd.cut(
#    df["square_feet"],
#    bins=[300, 500, 700, 1000, 1500, 2000, 2500, float("inf")],
#    labels=[300, 500, 700, 1000, 1500,2000,2500]
#)

df["bin_square_feet"] = pd.cut(
    df["square_feet"],
    bins=[300, 500, 700, 1000, 1500, 2000, 2500, float("inf")],
    labels=["0-300", "300-500", "500-700", "700-1000", "1000-1500","1500-2000","2000-2500"]
)
df.head(4)

,id,category,title,bathrooms,bedrooms,currency,fee,has_photo,pets_allowed,price,...,Wood Floors,price_per_sqft,Stan,Region,Strefa czasowa,Stan rolniczy,Stan przemyslowy,Wazny turystycznie,charakter,bin_square_feet
0,5668640009,housing/rent/apartment,One BR 507 & 509 Esplanade,1.0,1.0,USD,No,Thumbnail,Cats,2195.0,...,0,4.049815,California,West,PST,Tak,Tak,Tak,Roln Przem Turyst,300-500
1,5668639818,housing/rent/apartment,Three BR 146 Lochview Drive,1.5,3.0,USD,No,Thumbnail,"Cats,Dogs",1250.0,...,0,0.833333,Virginia,South,EST,Nie,Tak,Tak,Przem Turyst,700-1000
2,5668639686,housing/rent/apartment,Three BR 3101 Morningside Drive,2.0,3.0,USD,No,Thumbnail,NaN,1395.0,...,0,0.845455,North Carolina,South,EST,Tak,Tak,Tak,Roln Przem Turyst,1000-1500
3,5668639659,housing/rent/apartment,Two BR 209 Aegean Way,1.0,2.0,USD,No,Thumbnail,"Cats,Dogs",1600.0,...,0,1.951220,California,West,PST,Tak,Tak,Tak,Roln Przem Turyst,500-700


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    df,
    x='bin_square_feet',
    kde=True,
    hue='Region',
    alpha=.9,
    ax=axes[0]
)
axes[0].set_title('Histogram + KDE')

sns.kdeplot(
    df,
    x='square_feet',
    hue='Region',
    alpha=.9,
    multiple='stack',
    ax=axes[1]
)
axes[1].set_title('KDE stacked')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    df,
    x='Region',
    hue='charakter',
    alpha=.9,
    ax=axes[0],
    multiple='dodge',
    shrink=0.75
)
axes[0].set_title('Histogram + KDE')

sns.kdeplot(
    df,
    x='square_feet',
    hue='charakter',
    alpha=.9,
    multiple='stack',
    ax=axes[1]
)
axes[1].set_title('KDE stacked')

plt.tight_layout()
plt.show()

In [ ]:
df1 = df[(df["square_feet"] <= 2500) ]
fig = px.histogram(df1, x="charakter", color="Region")
fig.show()

In [ ]:
# wykres na którym na osi x będzie charakter, na osi y przedział_powierzchni, 
# kolorem określony stan, a wielkość kropki ma obrazować ilość linii w pliku odpowiadających kropce.

# zliczenie rekordów dla każdej kombinacji
agg = (
    df.groupby(
        ["charakter", "bin_square_feet", "Region"]
    )
    .size()
    .reset_index(name="liczba")
)

#print(agg.head())

# wykres

fig = px.scatter(
    agg,
    x="charakter",
    y="bin_square_feet",
    color="Region",
    size="liczba",
    size_max=60,
    hover_data=["liczba"],
    labels={
        "charakter": "Charakter regionu",
        "bin_square_feet": "Przedział powierzchni (stóp²)"
    },
    title="Liczba ofert wg charakteru, przedziału powierzchni i Regionu USA"
)

#fig.update_layout(
#    width=1200,
#    height=800
#)

fig.show()
